# Knowledge graph construction (AIVI vs. HI)

Builds the heterogeneous knowledge graph from user comments using LangChain's
`LLMGraphTransformer` with OpenAI `gpt-4o-mini` to extract head-relation-tail
triples, writes them to Neo4j, then exports the AIVI and HI subgraph edge
lists consumed by `graph_analysis.R`.

In [ ]:
!pip install --upgrade --quiet langchain langchain-community langchain-openai langchain-experimental langchain-neo4j neo4j pandas python-dotenv

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_core.documents import Document
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

In [ ]:
# Load credentials from environment (see .env_example)
load_dotenv()

PTH = os.getenv("PTH")
x = os.getenv("user_profile")

NEO4J_USERNAME = os.getenv("NEO4J_USER")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
NEO4J_URI = os.getenv("NEO4J_URI_" + x)
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD_" + x)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Load AIVI/HI comment data and prepare documents

In [ ]:
# Load into DataFrame
df = pd.read_csv(os.path.join(PTH,'AIVI_HI.csv'))
df['id'] = range(1, len(df) + 1)
df = df[df['posts.comments.text'].notna()]
df

In [ ]:
user_comments = df[df['text.comments.text'].notna()].copy()
user_comments.loc[:, 'user_or_influencer'] = 'USER'

influencers_posts = df[df['posts.comments.text'].notna() & df['text.comments.text'].isna()].copy()
influencers_posts.loc[:, 'user_or_influencer'] = 'INFLUENCER'

In [ ]:
user_comments = user_comments[['id', 
                                        'PROFILE.url',
                                        'user_type',
                                        'pot.comment.likes_count', 
                                        'posts.time',
                                        'posts.comments.user', 
                                        'text.comments.text',
                                        'user_or_influencer',                                       
                                        ]]

user_comments.columns = ['id',
                        'PROFILE.url',
                       'user_type',
                       'likes_count', 
                       'posting_time',
                       'posting_user', 
                       'text',
                       'user_or_influencer']


influencers_posts = influencers_posts[['id', 
                                         'PROFILE.url',
                                        'user_type',
                                        'posts.likes_count', 
                                        'posts.time',
                                        'posts.comments.user', 
                                        'posts.comments.text', 
                                        'user_or_influencer',                                       
                                        ]]

influencers_posts.columns = ['id',
                        'PROFILE.url',
                       'user_type',
                       'likes_count', 
                       'posting_time',
                       'posting_user', 
                       'text',
                       'user_or_influencer']


In [ ]:
# Combine the two DataFrames and ignore column names to avoid conflicts
combined_df = pd.concat([user_comments, influencers_posts], ignore_index=True)

# Remove rows with missing or empty text
combined_df = combined_df[combined_df['text'].notna() & (combined_df['text'].str.strip() != '')]

In [ ]:
documents = []
for index, row in combined_df.iterrows():
    doc = Document(
      page_content= row['text'],
      metadata={
           'id' : row['id'],
           'user_type' : row['user_type'],
           'likes_count' : row['likes_count'],
           'posting_time' : row['posting_time'],
           'posting_user' : row['posting_user'],
           'user_or_influencer' : row['user_or_influencer']
      }
    )
    documents.append(doc)

## Extract entities and relations with gpt-4o-mini

In [ ]:
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
    openai_api_key=OPENAI_API_KEY,
)

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[],
    allowed_relationships=[],
)

graph_documents = llm_transformer.convert_to_graph_documents(documents)

## Write the knowledge graph to Neo4j

In [ ]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

In [ ]:
# Storing to graph database in NEO4J
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

## Export the AIVI and HI subgraph edge lists

In [ ]:
# Cypher query to get internal edges of the subgraph connected to HUMAN and AIVI influencers (Document nodes)

cypher_query_HI = """
MATCH (d:Document {user_type: 'HUMAN'})
WITH collect(d) AS docs
UNWIND docs AS d
MATCH (d)--(n)  // collect connected nodes
WITH collect(DISTINCT d) + collect(DISTINCT n) AS S
UNWIND S AS node
MATCH (node)-[r]-(other)
WHERE other IN S
RETURN DISTINCT startNode(r).id AS source, endNode(r).id AS target, type(r) AS relation
"""


cypher_query_AIVI = """
MATCH (d:Document {user_type: 'AI'})
WITH collect(d) AS docs
UNWIND docs AS d
MATCH (d)--(n)  // collect connected nodes
WITH collect(DISTINCT d) + collect(DISTINCT n) AS S
UNWIND S AS node
MATCH (node)-[r]-(other)
WHERE other IN S
RETURN DISTINCT startNode(r).id AS source, endNode(r).id AS target, type(r) AS relation
"""

# Function to extract data
def extract_edgelist(uri, user, password, database, query):
    driver = GraphDatabase.driver(uri, auth=(user, password))
    with driver.session(database=database) as session:
        results = session.execute_read(lambda tx: tx.run(query).data())
    driver.close()
    return pd.DataFrame(results)

# Run the extraction
df_edgelist_HI = extract_edgelist(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE, cypher_query_HI)

df_edgelist_AIVI = extract_edgelist(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE, cypher_query_AIVI)

# Save to CSV or display
df_edgelist_HI.to_csv(os.path.join(PTH, "HI_subgraph_edgelist.csv"), index=False)
df_edgelist_AIVI.to_csv(os.path.join(PTH, "AIVI_subgraph_edgelist.csv"), index=False)